# Getting started with GUISkinDose
This notebook demonstrates the GUISkinDose Python API for estimating peak skin dose and creating dose maps from fluoroscopic RDSR data.

For a guided workflow, launch the GUI with `python -m guiskindose --mode gui`. Use this notebook when you want to script the same analysis from Python.

The latest version of the tutorial is available on GitHub at [https://github.com/kgrizz-git/GUISkinDose/blob/main/docs/source/getting_started/getting_started.ipynb](https://github.com/kgrizz-git/GUISkinDose/blob/main/docs/source/getting_started/getting_started.ipynb).

PyPI: [https://pypi.org/project/guiskindose/](https://pypi.org/project/guiskindose/)

Code repository: [https://github.com/kgrizz-git/GUISkinDose](https://github.com/kgrizz-git/GUISkinDose)

Documentation: [https://guiskindose.readthedocs.io/en/latest/](https://guiskindose.readthedocs.io/en/latest/)


## PART I: Settings 

This tutorial renders interactive plots directly in the notebook. To view them in a separate browser tab, uncomment the following two lines and set `settings.plot.notebook_mode` to `False` when running GUISkinDose.


In [ ]:
# import plotly.io as pio
# pio.renderers.default = "browser"

Start by importing `PyskindoseSettings`, the current public settings class, and the helpers used in this tutorial:

In [ ]:
from guiskindose import (
    PyskindoseSettings,
    load_settings_example_json,
    print_available_human_phantoms,
    get_path_to_example_rdsr_files,
    print_example_rdsr_files,
)
from guiskindose.main import main

Load the bundled `settings_example.json` template, then adjust its settings before calculating dose:

In [ ]:
# Parse the settings to a setting class:
settings_json = load_settings_example_json()
settings = PyskindoseSettings(settings=settings_json)

The settings are now available on `settings`. To inspect their current values, call `settings.print_parameters()`:

In [ ]:
settings.print_parameters()

`settings` groups general, phantom, plot, and normalization settings.

Access general settings directly, such as `settings.mode` and `settings.k_tab_val`. Access grouped settings through their section, for example `settings.phantom.patient_orientation`.

Use the corresponding `__doc__` attribute to inspect a settings class's available options.

Uncomment any of the following lines for details about a settings section.


In [ ]:
# print(settings.__doc__)  # uncomment this line to read about settings.general
# print(settings.phantom.__doc__) # uncomment this line to read about settings.phantom
# print(settings.phantom.patient_offset.__doc__)  # uncomment this line to read about settings.phantom.patient_offset
# print(settings.phantom.dimension.__doc__) # uncomment this line to read about settings.phantom.dimension
print(settings.plot.__doc__) # uncomment this line to read about settings.plot
# print(settings.normalization_settings.__doc__) # uncomment this line to read about settings.normalisation_settings

## PART II: Setup of the skin dose calculation geometry

Use GUISkinDose's `plot_setup` mode to inspect the phantom and its position on the patient support table.

Modify the phantom position with the translation parameters in `settings.phantom.patient_offset`.


In [ ]:
settings = PyskindoseSettings(settings=load_settings_example_json())
settings.mode = "plot_setup"
settings.phantom.model = "cylinder"

# Uncomment any of the following lines to translate the position of the phantom:
settings.phantom.patient_offset.d_lat = -20
# settings.phantom.patient_offset.d_lon = +10
# settings.phantom.patient_offset.d_ver = -10

main(settings=settings)

The plot shows the phantom on the support table with the X-ray source, beam, and image receptor in their standard position (`Ap1 = Ap2 = 0`).

Select a `"plane"`, `"cylinder"`, or `"human"` phantom with `settings.phantom.model`. For a human phantom, also set `settings.phantom.human_mesh`.

For plane and cylinder phantoms, adjust dimensions in `settings.phantom.dimension`. You can also change the patient support table and pad dimensions there.

In [ ]:
settings = PyskindoseSettings(settings=load_settings_example_json())
settings.mode = "plot_setup"

# Select an elliptical cylinder with length 150 cm and a 
# radius of[20, 10] cm
# settings.phantom.model = "cylinder"
# settings.phantom.dimension.cylinder_length = 150
# settings.phantom.dimension.cylinder_radii_a = 20
# settings.phantom.dimension.cylinder_radii_b = 10

# Select a planar phantom with length 120 cm and width 40 cm
# settings.phantom.model = "plane"
# settings.phantom.dimension.plane_length = 120
# settings.phantom.dimension.plane_width = 40

# Select a human phantom
settings.phantom.model = "human"
settings.phantom.human_mesh = "hudfrid"

main(settings=settings)

Use `print_available_human_phantoms()` to list the bundled human phantoms. To choose another one, replace `hudfrid` in the previous cell with a listed name and rerun it.


In [ ]:
print_available_human_phantoms()

After selecting the phantom and its position, inspect an RDSR before calculating dose.

## PART III: Load and examine an RDSR file

GUISkinDose includes these bundled example RDSR files:

In [ ]:
print_example_rdsr_files()

Use `plot_procedure` to inspect the orientation and size of the X-ray beam for each irradiation event. The plot slider lets you step through the events.

For large procedures, omit the patient phantom from this interactive plot by setting `settings.plot.max_events_for_patient_inclusion` below the event count.

The next cell loads a bundled example procedure and visualizes it with `plot_procedure`.

In [ ]:
settings = PyskindoseSettings(settings=load_settings_example_json())
settings.mode = "plot_procedure"
settings.phantom.model = "cylinder"

rdsr_data_dir = get_path_to_example_rdsr_files()

# You can set the maximum number of irradiation events for including the
# phantom in the plot. Here, we set it to 0 to reduce memory use.
settings.plot.max_events_for_patient_inclusion = 0

# Change this path to use your own RDSR file
# N.B: If your are using a windows OS, you need to set the path as r raw string,
# for example: 
# selected_rdsr_filepath = Path(r'c:\rdsr_files\file_name.dcm')
selected_rdsr_filepath = rdsr_data_dir / "siemens_axiom_example_procedure.dcm"

main(settings=settings, file_path=selected_rdsr_filepath)

To analyze your own RDSR, pass its path as `file_path`.

## PART IV: Calculate and plot a dose map

Once you have selected the phantom, positioned it on the support table, and reviewed the RDSR, you can calculate and visualize dose.

Set `settings.mode` to `"calculate_dose"` to calculate the dose estimate and visualize the result as a dose map.

The interactive plot combines RDSR information with the available correction factors. Rotate the phantom and hover over it to inspect estimated skin dose.

In [ ]:
settings = PyskindoseSettings(settings=load_settings_example_json())
settings.mode = "calculate_dose"
settings.output_format = 'html' # set output format to html
settings.plot.plot_dosemap = True # enable dosemap plot
settings.phantom.model = "human"
settings.phantom.human_mesh = "hudfrid"

settings.phantom.patient_offset.d_lat = -35
 
main(settings=settings, file_path=rdsr_data_dir / "siemens_axiom_example_procedure.dcm")


To inspect, analyze, or save calculation results in Python, set `settings.output_format` to `"dict"` or `"json"`:

In [ ]:
settings = PyskindoseSettings(settings=load_settings_example_json())
settings.mode = "calculate_dose"
settings.output_format = 'dict'

rdsr_data_dir = get_path_to_example_rdsr_files()
output = main(settings=settings, file_path=rdsr_data_dir / "siemens_axiom_example_procedure.dcm")

print(f'estimated psd {output["psd"].round(1)} mGy')

The dictionary contains these top-level fields from the calculation:

- **schema_version**
    - Version of the exported dictionary schema
- **psd**, **air_kerma**, and **air_kerma_corrected**
    - Summary dose values
- **events**
    - Per-event input and calculated values
- **table**
    - Patient support table geometry
- **pad**
    - Patient support pad geometry
- **patient**
    - Phantom details, orientation, and offsets
- **dose_map**
    - Sparse skin-patch indices and estimated dose values
- **corrections**
    - Sparse correction indices plus backscatter, medium, table, inverse-square-law, kerma, corrected-kerma, and kerma-meter values


## Adjusting correction factors

One correction you can adjust is the estimated transmission through the patient support table and pad. Set `k_tab_val` to a value greater than 0 and at most 1, where 1 means no attenuation by the table or pad.

Lower the table and pad transmission factor to compare the resulting dose map.

In [ ]:
settings = PyskindoseSettings(settings=load_settings_example_json())
settings.mode = "calculate_dose"
settings.output_format = 'html' # set output format to html
settings.plot.plot_dosemap = True # enable dosemap plot
settings.phantom.model = "human"
settings.phantom.human_mesh = "hudfrid"

settings.phantom.patient_offset.d_lat = -35
settings.k_tab_val = 0.5
main(settings=settings, file_path=rdsr_data_dir / "siemens_axiom_example_procedure.dcm")


In this example, the dose on the back of the phantom decreases because the beam passes through the table and pad. Dose on the side remains unchanged when the beam does not pass through them.


## Further development and contributions

Contributions and collaboration are welcome. See [the contribution guide](../user/contribute.html).

For problems using GUISkinDose, open an issue on the [project issue tracker](https://github.com/kgrizz-git/GUISkinDose/issues/).